In [1]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.

In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [5]:
# Scenario: Customer Support Chatbot (Question-Based)
# Imagine a company has deployed a chatbot that answers customer
#  questions by calling the Groq API. The workflow is modeled as a
#  graph of states, where each customer query flows through nodes until
#   a response is delivered.

# 1. State Definition
# The chatbot maintains a notebook-like state:
# - question → The customer’s query.
# - answer → The response generated by Groq.
# - history → A log of all past questions.

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    groq_api_key = os.getenv("GROQ_API_KEY")

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    # Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            # IMPORTANT: The list of supported models by Groq API changes frequently.
            # Please refer to the official Groq documentation (https://console.groq.com/docs/models)
            # to find a currently active and supported model name and replace 'YOUR_GROQ_MODEL_NAME_HERE' below.
            # Examples of often available models include 'llama3-8b-8192' or 'llama3-70b-8192',
            # but these can also become decommissioned.
            "model": "llama-3.1-8b-instant",   # Changed to a currently active model
            "messages": [{"role": "user", "content": q}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    # Extract answer from Groq response
    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    # Initial state
    state = {"question": "What are your store hours?", "answer": "", "history": []}

    # Run the graph
    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])

Bot says: I'm a large language model, I don't have a physical store. I exist solely as a digital entity, and I'm available 24/7 to assist with questions and tasks. You can interact with me at any time, from anywhere with an internet connection.


In [ ]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each learner:
# - topic → The subject the learner is studying (e.g., "Photosynthesis").
# - flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
# - progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

# 2. Workflow (Graph of States)
# Each learner interaction flows through nodes until a flashcard is delivered:
# - Input Node
# - Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
# - State updates: topic = "cell biology"
# - Generation Node (Groq API)
# - Groq generates a flashcard:
# - flashcard.question = "What organelle is known as the powerhouse of the cell?"
# - flashcard.answer = "Mitochondria"
# - Response Node
# - Assistant presents the flashcard question to the learner.
# - Evaluation Node
# - Learner responds with their answer.
# - Assistant checks correctness and updates progress.
# - History Node
# - Logs the flashcard attempt:
# - progress = [{question, learner_answer, correct/incorrect}]

In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import random

# 1. State
class StudyState(TypedDict):
    topic: str
    flashcard: Dict
    user_answer: str
    progress: List[Dict]

# 2. Static Flashcards DB
FLASHCARDS_DB = {
    "cell biology": [
        {"question": "What is the powerhouse of the cell?", "answer": "mitochondria"},
        {"question": "What controls cell activities?", "answer": "nucleus"},
    ],
    "photosynthesis": [
        {"question": "Where does photosynthesis occur?", "answer": "chloroplast"},
        {"question": "What gas is released?", "answer": "oxygen"},
    ]
}

# 3. Nodes
def input_node(state: StudyState):
    return state

def generate_flashcard(state: StudyState):
    topic = state["topic"].lower()
    cards = FLASHCARDS_DB.get(topic, [])
    
    if not cards:
        card = {"question": "No data available", "answer": "N/A"}
    else:
        card = random.choice(cards)

    return {"flashcard": card}

def ask_question(state: StudyState):
    print("\n📘 Question:", state["flashcard"]["question"])
    user_ans = input("Your Answer: ")
    return {"user_answer": user_ans}

def evaluate(state: StudyState):
    correct_ans = state["flashcard"]["answer"].lower()
    user_ans = state["user_answer"].lower()

    is_correct = user_ans == correct_ans

    print("✅ Correct!" if is_correct else f"❌ Wrong! Correct answer: {correct_ans}")

    return {
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "user_answer": user_ans,
            "correct": is_correct
        }]
    }

# 4. Graph
graph = StateGraph(StudyState)

graph.add_node("input", input_node)
graph.add_node("generate", generate_flashcard)
graph.add_node("ask", ask_question)
graph.add_node("evaluate", evaluate)

graph.set_entry_point("input")
graph.add_edge("input", "generate")
graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", END)

app = graph.compile()

# 5. Run
state = {
    "topic": "cell biology",
    "flashcard": {},
    "user_answer": "",
    "progress": []
}

result = app.invoke(state)

print("\n📊 Progress:", result["progress"])


📘 Question: What controls cell activities?


Your Answer:  nucleus


✅ Correct!

📊 Progress: [{'question': 'What controls cell activities?', 'user_answer': 'nucleus', 'correct': True}]


In [4]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each learner:
# - topic → The subject the learner is studying (e.g., "Photosynthesis").
# - flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
# - progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

# 2. Workflow (Graph of States)
# Each learner interaction flows through nodes until a flashcard is delivered:
# - Input Node
# - Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
# - State updates: topic = "cell biology"
# - Generation Node (Groq API)
# - Groq generates a flashcard:
# - flashcard.question = "What organelle is known as the powerhouse of the cell?"
# - flashcard.answer = "Mitochondria"
# - Response Node
# - Assistant presents the flashcard question to the learner.
# - Evaluation Node
# - Learner responds with their answer.
# - Assistant checks correctness and updates progress.
# - History Node
# - Logs the flashcard attempt:
# - progress = [{question, learner_answer, correct/incorrect}]


# With Api

In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 1. State
class StudyState(TypedDict):
    topic: str
    flashcard: Dict
    user_answer: str
    progress: List[Dict]

# 2. Generate Flashcard (Groq)
def generate_flashcard(state: StudyState):
    api_key = os.getenv("GROQ_API_KEY")

    prompt = f"""
    Generate one flashcard for topic: {state['topic']}.
    Format strictly:
    Question: <question>
    Answer: <answer>
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    # Parse output
    lines = content.split("\n")
    question = lines[0].replace("Question:", "").strip()
    answer = lines[1].replace("Answer:", "").strip()

    return {"flashcard": {"question": question, "answer": answer}}

# 3. Ask Question
def ask_question(state: StudyState):
    print("\n📘 Question:", state["flashcard"]["question"])
    user_ans = input("Your Answer: ")
    return {"user_answer": user_ans}

# 4. Evaluate
def evaluate(state: StudyState):
    correct_ans = state["flashcard"]["answer"].lower()
    user_ans = state["user_answer"].lower()

    is_correct = correct_ans in user_ans  # flexible matching

    print("✅ Correct!" if is_correct else f"❌ Wrong! Correct: {correct_ans}")

    return {
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "user_answer": user_ans,
            "correct": is_correct
        }]
    }

# 5. Graph
graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("ask", ask_question)
graph.add_node("evaluate", evaluate)

graph.set_entry_point("generate")
graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", END)

app = graph.compile()

# 6. Run
state = {
    "topic": "photosynthesis",
    "flashcard": {},
    "user_answer": "",
    "progress": []
}

result = app.invoke(state)

print("\n📊 Progress:", result["progress"])


📘 Question: What process do plants use to convert sunlight into energy?


Your Answer:  No idea


❌ Wrong! Correct: photosynthesis

📊 Progress: [{'question': 'What process do plants use to convert sunlight into energy?', 'user_answer': 'no idea', 'correct': False}]


In [6]:
# Scenario: AI-Powered Project Tracker (Task-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each project:
# - task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
# - status → The current state of the task (e.g., "in progress", "completed", "blocked").
# - log → A history of all updates, including who made them and when.

# 2. Workflow (Graph of States)
# Each project update flows through nodes until the task status is refreshed:
# - Input Node
# - Team member submits an update (e.g., "Report draft completed").
# - State updates: task = "Q1 financial report"
# - Processing Node (Groq API)
# - Groq interprets the update and assigns a status:
# - status = "completed"
# - Response Node
# - Assistant confirms the update back to the team:
# - "Task Q1 financial report marked as completed."
# - History Node
# - Logs the update:
# - log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

In [7]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
from datetime import datetime

# 1. State
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    log: List[Dict]

# 2. Nodes
def input_node(state: ProjectState):
    return state

def process_update(state: ProjectState):
    update = state["update"].lower()

    # Rule-based classification
    if "completed" in update or "done" in update:
        status = "completed"
    elif "blocked" in update or "issue" in update:
        status = "blocked"
    else:
        status = "in progress"

    return {"status": status}

def response_node(state: ProjectState):
    msg = f"✅ Task '{state['task']}' marked as {state['status']}."
    print(msg)
    return {}

def log_node(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "timestamp": str(datetime.now())
    }

    return {"log": state["log"] + [entry]}

# 3. Graph
graph = StateGraph(ProjectState)

graph.add_node("input", input_node)
graph.add_node("process", process_update)
graph.add_node("response", response_node)
graph.add_node("log", log_node)

graph.set_entry_point("input")
graph.add_edge("input", "process")
graph.add_edge("process", "response")
graph.add_edge("response", "log")
graph.add_edge("log", END)

app = graph.compile()

# 4. Run
state = {
    "task": "Q1 financial report",
    "update": "Report draft completed",
    "status": "",
    "log": []
}

result = app.invoke(state)

print("\n📊 Log:", result["log"])

✅ Task 'Q1 financial report' marked as completed.

📊 Log: [{'task': 'Q1 financial report', 'update': 'Report draft completed', 'status': 'completed', 'timestamp': '2026-03-20 11:54:32.429889'}]


In [8]:
# WITH API

In [9]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
from datetime import datetime
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 1. State
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    log: List[Dict]

# 2. Groq Processing Node
def process_update(state: ProjectState):
    api_key = os.getenv("GROQ_API_KEY")

    prompt = f"""
    You are a project tracker AI.

    Task: {state['task']}
    Update: {state['update']}

    Classify task status into ONLY one:
    - completed
    - in progress
    - blocked

    Respond ONLY with one word.
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    data = response.json()
    status = data["choices"][0]["message"]["content"].strip().lower()

    return {"status": status}

# 3. Response Node
def response_node(state: ProjectState):
    msg = f"📌 Task '{state['task']}' marked as {state['status']}."
    print(msg)
    return {}

# 4. Log Node
def log_node(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "timestamp": str(datetime.now())
    }

    return {"log": state["log"] + [entry]}

# 5. Graph
graph = StateGraph(ProjectState)

graph.add_node("process", process_update)
graph.add_node("response", response_node)
graph.add_node("log", log_node)

graph.set_entry_point("process")
graph.add_edge("process", "response")
graph.add_edge("response", "log")
graph.add_edge("log", END)

app = graph.compile()

# 6. Run
state = {
    "task": "Q1 financial report",
    "update": "We are stuck due to missing data from finance team",
    "status": "",
    "log": []
}

result = app.invoke(state)

print("\n📊 Log:", result["log"])

📌 Task 'Q1 financial report' marked as blocked..

📊 Log: [{'task': 'Q1 financial report', 'update': 'We are stuck due to missing data from finance team', 'status': 'blocked.', 'timestamp': '2026-03-20 11:55:11.084770'}]


In [10]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


In [11]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# Node 1: Search for information
def search_web(state: ResearchState):
    print(f"\u001f Searching: {state['topic']}")
    # Simulate web search results
    new_results = [
        f"Article 1 about {state['topic']}",
        f"Article 2 about {state['topic']}",
    ]
    return {"search_results": state["search_results"] + new_results,
            "steps_done": state["steps_done"] + 1}

# Node 2: Analyze the results
def analyze_results(state: ResearchState):
    print(f"\u001f Analyzing {len(state['search_results'])} results")
    analysis = f"Key insights from {len(state['search_results'])} sources"
    return {"analysis": analysis,
            "steps_done": state["steps_done"] + 1}

# Node 3: Summarize
def summarize(state: ResearchState):
    print("\u001f\u001f Generating summary...")
    summary = f"Summary: {state['analysis']}"
    return {"summary": summary}

# Node 4: Check if we need more research
def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back!
    return END                 # Done

# Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", END: "summarize"})
g.add_edge("summarize", END)

app = g.compile()
result = app.invoke(
    {
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    }
)
print(result["summary"])

 Searching: Quantum Computing
 Analyzing 2 results
 Searching: Quantum Computing
 Analyzing 4 results
 Generating summary...
Summary: Key insights from 4 sources


In [14]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
import os
from dotenv import load_dotenv

load_dotenv()


# 1. Define State
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"): # Changed model to a known working one
    groq_api_key = os.getenv('Groq_API_KEY')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def search_web(state: ResearchState):
    print(f"🔍 Searching: {state['topic']}")
    # Call Groq to generate snippets
    new_results = [
        groq_call(f"Give me a short article snippet about {state['topic']}"),
        groq_call(f"Give me another snippet about {state['topic']}")
    ]
    results = state["search_results"] + new_results
    return {
        "search_results": results,
        "steps_done": state["steps_done"] + 1
    }

def analyze_results(state: ResearchState):
    print(f"📊 Analyzing {len(state['search_results'])} results")
    joined_results = "\n".join(state["search_results"])
    analysis = groq_call(f"Analyze these sources and extract key insights:\n{joined_results}")
    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

def summarize(state: ResearchState):
    print("✍️ Generating summary...")
    summary = groq_call(f"Summarize this analysis in simple terms:\n{state['analysis']}")
    return {"summary": summary}

def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back until enough results
    return "summarize"        # Once we have 3+, move to summary

# 4. Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", "summarize": "summarize"})
g.add_edge("summarize", END)

# 5. Run the graph
if __name__ == "__main__":
    app = g.compile()
    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [], "analysis": "",
        "summary": "", "steps_done": 0
    })
    print("\n✅ Final Summary:\n", result["summary"])

🔍 Searching: Quantum Computing
📊 Analyzing 2 results
🔍 Searching: Quantum Computing
📊 Analyzing 4 results
✍️ Generating summary...

✅ Final Summary:
 Here's a simple summary of the analysis:

**What is Quantum Computing?**

- Quantum Computing is a new way of processing information that uses tiny particles called qubits. These qubits can exist in multiple states at the same time, which makes them very powerful.
- Quantum Computing has the potential to make breakthroughs in medicine, data security, climate modeling, and finance.

**Challenges and Limitations**

- Quantum Computing is still in its early stages and there are many challenges to overcome, such as the fragility of qubits and interference from the environment.
- Error correction is a big issue, as it's hard to make sure the qubits are working correctly.
- We also need to figure out how to scale up Quantum Computing to millions of qubits.

**Advantages and Applications**

- Quantum Computing can be used to simulate complex sys

In [15]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.

In [16]:
# Without Api

In [17]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

# 1. State
class HealthState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int

# 2. Observation Node (rule-based)
def observe(state: HealthState):
    symptom = state["symptom"].lower()
    obs = state["observations"]

    if "cough" in symptom:
        new_obs = [
            "Could be due to cold or flu",
            "May be caused by allergies",
            "Persistent cough may indicate infection"
        ]
    else:
        new_obs = ["General symptom, needs more observation"]

    # Add one observation at a time (simulate loop)
    if state["steps_done"] < len(new_obs):
        obs.append(new_obs[state["steps_done"]])

    return {
        "observations": obs,
        "steps_done": state["steps_done"] + 1
    }

# 3. Analysis Node
def analyze(state: HealthState):
    analysis = " | ".join(state["observations"])
    return {"analysis": analysis}

# 4. Conditional Check
def check_condition(state: HealthState):
    if len(state["observations"]) < 3:
        return "observe"
    else:
        return "recommend"

# 5. Recommendation Node
def recommend(state: HealthState):
    rec = "If symptoms persist for more than 2 weeks, consult a doctor."
    print("\n💡 Recommendation:", rec)
    return {"recommendation": rec}

# 6. Graph
graph = StateGraph(HealthState)

graph.add_node("observe", observe)
graph.add_node("analyze", analyze)
graph.add_node("recommend", recommend)

graph.set_entry_point("observe")

graph.add_edge("observe", "analyze")
graph.add_conditional_edges("analyze", check_condition)
graph.add_edge("recommend", END)

app = graph.compile()

# 7. Run
state = {
    "symptom": "persistent cough",
    "observations": [],
    "analysis": "",
    "recommendation": "",
    "steps_done": 0
}

result = app.invoke(state)

print("\n🧾 Observations:", result["observations"])
print("📊 Analysis:", result["analysis"])


💡 Recommendation: If symptoms persist for more than 2 weeks, consult a doctor.

🧾 Observations: ['Could be due to cold or flu', 'May be caused by allergies', 'Persistent cough may indicate infection']
📊 Analysis: Could be due to cold or flu | May be caused by allergies | Persistent cough may indicate infection


In [18]:
# With api

In [19]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 1. State
class HealthState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int

# 2. Observation Node (Groq)
def observe(state: HealthState):
    api_key = os.getenv("GROQ_API_KEY")

    prompt = f"""
    Symptom: {state['symptom']}

    Give ONE short possible observation or cause.
    Keep it simple and non-diagnostic.
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    data = response.json()
    obs_text = data["choices"][0]["message"]["content"].strip()

    return {
        "observations": state["observations"] + [obs_text],
        "steps_done": state["steps_done"] + 1
    }

# 3. Analysis Node
def analyze(state: HealthState):
    combined = " | ".join(state["observations"])

    analysis = f"Possible insights: {combined}"

    return {"analysis": analysis}

# 4. Conditional Node
def check_condition(state: HealthState):
    if len(state["observations"]) < 3:
        return "observe"
    else:
        return "recommend"

# 5. Recommendation Node (Groq)
def recommend(state: HealthState):
    api_key = os.getenv("GROQ_API_KEY")

    prompt = f"""
    Symptom: {state['symptom']}
    Observations: {state['observations']}

    Give a simple, safe, non-medical recommendation.
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    data = response.json()
    rec = data["choices"][0]["message"]["content"].strip()

    print("\n💡 Recommendation:", rec)

    return {"recommendation": rec}

# 6. Graph
graph = StateGraph(HealthState)

graph.add_node("observe", observe)
graph.add_node("analyze", analyze)
graph.add_node("recommend", recommend)

graph.set_entry_point("observe")

graph.add_edge("observe", "analyze")
graph.add_conditional_edges("analyze", check_condition)
graph.add_edge("recommend", END)

app = graph.compile()

# 7. Run
state = {
    "symptom": "persistent cough",
    "observations": [],
    "analysis": "",
    "recommendation": "",
    "steps_done": 0
}

result = app.invoke(state)

print("\n🧾 Observations:", result["observations"])
print("📊 Analysis:", result["analysis"])


💡 Recommendation: Based on the symptoms and observations, I recommend the following safe, non-medical suggestion:

**Recommendation:** Limit exposure to airborne irritants and allergens.

**Actions to take:**

1. Avoid smoking and secondhand smoke.
2. Stay away from polluted areas or areas with high levels of air pollution.
3. Avoid strong chemicals, fumes, or fragrances.
4. Wear a mask when outdoors in polluted conditions.
5. Vacuum frequently and dust less often to reduce airborne particles.
6. Consider using a HEPA air purifier to improve indoor air quality.
7. Keep an eye on pollen counts if allergies are a possibility.

**Remember:** This is not a substitute for a medical professional's advice. If your symptoms persist or worsen, consult a doctor for proper assessment and treatment.

🧾 Observations: ['Possible observation: exposure to airborne irritants.', 'The person may be exposed to airborne irritants or allergens.', 'Possible cause: Exposure to airborne particles or irritants

In [22]:
# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests

import os
from dotenv import load_dotenv

load_dotenv()

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = os.getenv('Groq_API_KEY')
    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def draft_email(state: EmailState):
    print(f"📝 Drafting email for task: {state['task']}")
    draft = groq_call(f"Draft a professional email regarding: {state['task']}")
    return {"draft": draft}

def human_review(state: EmailState):
    # Interrupt node: waits for human approval
    print(f"📧 Draft ready for review:\n\n{state['draft']}\n")
    return {}  # Pauses here until human updates 'approved'

def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}

# 4. Build Graph
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]  # Pause before review
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Draft email
app.invoke({"task": "Q3 Report", "draft": "", "approved": False}, thread)

# Step 2: Human reviews draft and resumes
app.invoke({"approved": True}, thread)

📝 Drafting email for task: Q3 Report
📝 Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': "Subject: Q3 [Year] Performance Report\n\nDear [Recipient's Name],\n\nI hope this email finds you well. As per our quarterly reporting requirements, I am writing to submit the Q3 [Year] Performance Report for your review and consideration.\n\nBelow are some key highlights of our performance during the third quarter:\n\n- **Revenue Growth:** Our revenue for the period between [start date] and [end date] was $[revenue figure], representing a [growth percentage]% increase compared to the same period last year.\n- **Expense Management:** We maintained a healthy expense ratio of [expense ratio]% for the quarter, demonstrating our commitment to efficient resource allocation.\n- **Key Milestones:** We successfully [mention key projects or achievements] during the quarter, which have contributed to our overall growth and success.\n\nThe full Q3 [Year] Performance Report, including financial statements, operational metrics, and strategic insights, is attached to 

In [23]:
# Scenario Question
# "Imagine you are designing an AI-powered assistant that drafts structured reports,
# pauses for human review, and then either publishes or rejects them. How would you structure the state and workflow graph to 
# ensure accountability and human oversight in the process?"

In [24]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List, Dict
from datetime import datetime
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 1. State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool
    reviewer: str
    status: str
    history: List[Dict]

# 2. Groq API
def groq_call(prompt: str):
    api_key = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise ValueError("GROQ_API_KEY not set")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    return response.json()["choices"][0]["message"]["content"]

# 3. Nodes

# Draft Node
def draft_email(state: EmailState):
    draft = groq_call(f"Draft a professional email for: {state['task']}")

    entry = {
        "action": "draft_generated",
        "timestamp": str(datetime.now())
    }

    return {
        "draft": draft,
        "status": "DRAFT",
        "history": state["history"] + [entry]
    }

# Review Node (HITL)
def review_node(state: EmailState):
    print("\n📧 Draft for Review:\n", state["draft"])
    return {}

# Decision Function
def decide(state: EmailState):
    return "send" if state.get("approved") else "reject"

# Send Node
def send_email(state: EmailState):
    entry = {
        "action": "sent",
        "reviewer": state["reviewer"],
        "timestamp": str(datetime.now())
    }

    print("✅ Email Sent")

    return {
        "status": "SENT",
        "history": state["history"] + [entry]
    }

# Reject Node
def reject_email(state: EmailState):
    entry = {
        "action": "rejected",
        "reviewer": state["reviewer"],
        "timestamp": str(datetime.now())
    }

    print("❌ Email Rejected")

    return {
        "status": "REJECTED",
        "history": state["history"] + [entry]
    }

# 4. Graph
graph = StateGraph(EmailState)

graph.add_node("draft", draft_email)
graph.add_node("review", review_node)
graph.add_node("send", send_email)
graph.add_node("reject", reject_email)

graph.set_entry_point("draft")

graph.add_edge("draft", "review")
graph.add_conditional_edges("review", decide)
graph.add_edge("send", END)
graph.add_edge("reject", END)

# 5. Checkpointer (for pause/resume)
memory = MemorySaver()

app = graph.compile(
    checkpointer=memory,
    interrupt_before=["review"]
)

# 6. Run
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Generate draft
app.invoke({
    "task": "Q3 Report",
    "draft": "",
    "approved": False,
    "reviewer": "",
    "status": "",
    "history": []
}, thread)

# Step 2: Human review input
app.invoke({
    "approved": True,
    "reviewer": "Manager"
}, thread)

{'task': 'Q3 Report',
 'draft': "Here's a sample professional email for the Q3 Report:\n\nSubject: Q3 Report 2023 and Key Performance Indicators (KPIs)\n\nDear [Recipient's Name],\n\nI am pleased to provide you with the Quarterly Report for the third quarter of [Year], which covers the period from [Date] to [Date]. This report outlines the company's performance, key accomplishments, and highlights the progress made towards our strategic objectives.\n\n**Summary of Q3 Performance:**\n\n- Revenue: $[Amount] (up [Percentage] from Q3 [Year - 1])\n- Net Income: $[Amount] (increase of [Percentage] from Q3 [Year - 1])\n- Key Performance Indicators (KPIs):\n + [KPI 1]: [Value]\n + [KPI 2]: [Value]\n + [KPI 3]: [Value]\n\n**Key Accomplishments:**\n\n1. [Briefly mention the key accomplishments of the quarter, such as new product launches, significant contracts, or notable milestones achieved.]\n2. [Include any notable successes or achievements, highlighting the impact on the company and its stak